In [4]:
#计算站点周围的4个格网坐标值 列名为 1 2 3 4
import math
import pandas as pd

def calc_grid_corners(lat, lon):
    lat_floor = math.floor(lat * 10) / 10
    lon_floor = math.floor(lon * 10) / 10

    lat_upper = lat_floor + 0.1
    lon_upper = lon_floor + 0.1

    # 四个角点 (未round前)
    corner1 = (lat_floor, lon_floor)
    corner2 = (lat_floor, lon_upper)
    corner3 = (lat_upper, lon_floor)
    corner4 = (lat_upper, lon_upper)

    # 四舍五入到1位小数
    c1 = (round(corner1[0], 1), round(corner1[1], 1))
    c2 = (round(corner2[0], 1), round(corner2[1], 1))
    c3 = (round(corner3[0], 1), round(corner3[1], 1))
    c4 = (round(corner4[0], 1), round(corner4[1], 1))
    return c1, c2, c3, c4

def add_grid_corners_to_csv(csv_path):
    df = pd.read_csv(csv_path)

    lat_col = "latitude_installation"
    lon_col = "longitude_installation"

    corner1_list = []
    corner2_list = []
    corner3_list = []
    corner4_list = []

    for _, row in df.iterrows():
        lat_val = row[lat_col]
        lon_val = row[lon_col]
        
        c1, c2, c3, c4 = calc_grid_corners(lat_val, lon_val)

        # 将它们转换成字符串格式 "(x.x, y.y)"
        c1_str = f"({c1[0]:.1f}, {c1[1]:.1f})"
        c2_str = f"({c2[0]:.1f}, {c2[1]:.1f})"
        c3_str = f"({c3[0]:.1f}, {c3[1]:.1f})"
        c4_str = f"({c4[0]:.1f}, {c4[1]:.1f})"

        corner1_list.append(c1_str)
        corner2_list.append(c2_str)
        corner3_list.append(c3_str)
        corner4_list.append(c4_str)

    # 新增四列(名为1,2,3,4)
    df["1"] = corner1_list
    df["2"] = corner2_list
    df["3"] = corner3_list
    df["4"] = corner4_list

    # 保存到CSV
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"已处理并写回: {csv_path}")

if __name__ == "__main__":
    csv_file = r"F:\TUB\greenlandProject\NEWdataverse_files\AWS_sites_metadata.csv"
    add_grid_corners_to_csv(csv_file)



已处理并写回: F:\TUB\greenlandProject\NEWdataverse_files\AWS_sites_metadata.csv


In [6]:
#合并所有格网的坐标 到列5
import pandas as pd

def expand_and_split_corners(input_csv, output_csv):
    """
    读取 input_csv，其中包含列 '1','2','3','4'（形如 '(66.4, -46.2)' 的坐标字符串）。
    将它们逐行展开：每个坐标 -> 一条新行，并拆分成新列 'latitude','longitude'（数值）。
    最后将结果保存到 output_csv。
    """
    # 1) 读取原 CSV
    df = pd.read_csv(input_csv)

    # 如果列名确实是数字字符串 "1","2","3","4"，就这样写：
    corner_cols = ["1", "2", "3", "4"]

    # 用来存放“展开+拆分”后的所有行
    expanded_records = []

    # 2) 遍历原表每一行
    for idx, row in df.iterrows():
        # 取出列 '1','2','3','4' 的字符串，例如 "(66.4, -46.2)"
        corners = []
        for col in corner_cols:
            coord_str = row.get(col, None)
            if not isinstance(coord_str, str):
                # 如果该列是 NaN 或其他类型，不处理
                continue
            
            # 去掉左右括号，按逗号拆分
            tmp = coord_str.strip("()").split(",")
            if len(tmp) != 2:
                # 格式异常：比如没有逗号或空值
                continue
            
            lat_str = tmp[0].strip()
            lon_str = tmp[1].strip()

            try:
                lat_val = float(lat_str)
                lon_val = float(lon_str)
                corners.append((lat_val, lon_val))
            except ValueError:
                # 如果无法解析为浮点数，则跳过
                pass

        # 如果 4 个角不全，则 corners 列表可能少于 4 个，也没关系
        # 3) 如果想按纬度->经度排序： 
        corners_sorted = sorted(corners, key=lambda c: (c[0], c[1]))

        # 4) 对每个坐标，复制原行，并拆分 lat, lon
        for (lat_val, lon_val) in corners_sorted:
            new_row = row.copy()
            new_row["latitude"] = lat_val
            new_row["longitude"] = lon_val
            expanded_records.append(new_row)

    # 5) 将收集到的“展开行”转为新的 DataFrame
    df_expanded = pd.DataFrame(expanded_records)

    # 如果不想保留原先的 '1','2','3','4' 列，可以直接 drop：
    # df_expanded.drop(columns=corner_cols, inplace=True, errors='ignore')

    # 6) 写出到新 CSV（避免覆盖原文件）
    df_expanded.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"展开与拆分完成，结果写入: {output_csv}")

if __name__ == "__main__":
    # 示例：输入与输出 CSV 路径
    input_path = r"F:\TUB\greenlandProject\NEWdataverse_files\AWS_sites_metadata.csv"
    output_path = r"F:\TUB\greenlandProject\NEWdataverse_files\AWS_sites_metadata_expanded.csv"

    expand_and_split_corners(input_path, output_path)



展开与拆分完成，结果写入: F:\TUB\greenlandProject\NEWdataverse_files\AWS_sites_metadata_expanded.csv


In [1]:
import pandas as pd

def filter_forecast_by_grid():
    # 1) 读取 grid.xlsx
    grid_file = r"F:\TUB\greenlandProject\NEWdataverse_files\grid.xlsx"
    grid_df = pd.read_excel(grid_file)  # 默认读取第一个工作表
    
    # 假设 grid_df 中确实有 'latitude' 和 'longitude' 两列
    # 如果列名不同，请自行更改
    grid_df = grid_df[['latitude', 'longitude']].drop_duplicates()
    
    # 2) 读取 forecastAlbedo_1995.csv
    forecast_file = r"F:\TUB\greenlandProject\ERA5LAND\forecastalbedo\forecastAlbedo_1995.csv"
    forecast_df = pd.read_csv(forecast_file)
    
    # 同样假设 forecast_df 中也有 'latitude' 和 'longitude' 两列
    # 如果列名不同，请自行更改
    
    # 3) 用 pandas.merge(inner join) 筛选出共用的 (lat,lon) 行
    #    how='inner' 表示只保留匹配上的部分
    merged_df = pd.merge(
        forecast_df, 
        grid_df, 
        on=['latitude','longitude'], 
        how='inner'
    )
    
    # 4) 写出匹配到的新文件
    output_file = r"F:\TUB\greenlandProject\ERA5LAND\forecastalbedo\1995grid.csv"
    merged_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print(f"已生成匹配文件：{output_file}, 匹配行数：{len(merged_df)}")

if __name__ == "__main__":
    filter_forecast_by_grid()


已生成匹配文件：F:\TUB\greenlandProject\ERA5LAND\forecastalbedo\1995grid.csv, 匹配行数：2124


In [2]:
#将原始包含所有坐标点的Albedo_1995.csv 转换为仅包含PROMICE周围兴趣点的GRID文件
#snowAlbedo_1995_grid.csv
import pandas as pd
import glob
import os

def filter_all_snows_by_grid():
    # 1) 读取 grid.xlsx
    grid_file = r"F:\TUB\greenlandProject\NEWdataverse_files\grid.xlsx"
    grid_df = pd.read_excel(grid_file)  # 默认读取第一个工作表
    # 假设 grid_df 中有 'latitude' 和 'longitude' 两列，如名称不同请自行调整
    grid_df = grid_df[['latitude', 'longitude']].drop_duplicates()

    # 2) 指定 snow 文件所在的文件夹
    snow_folder = r"F:\TUB\greenlandProject\ERA5LAND\snowalbedo"
    # 使用 glob 获取文件夹中所有 CSV 文件的完整路径
    snow_files = glob.glob(os.path.join(snow_folder, "*.csv"))

    if not snow_files:
        print("未找到任何 CSV 文件！")
        return

    # 3) 遍历每个 snow CSV 文件
    for snow_file in snow_files:
        try:
            snow_df = pd.read_csv(snow_file)
        except Exception as e:
            print(f"读取文件 {snow_file} 出错：{e}")
            continue

        # 检查 snow_df 是否包含 'latitude' 和 'longitude' 两列
        if not {'latitude', 'longitude'}.issubset(snow_df.columns):
            print(f"文件 {snow_file} 不包含所需的 'latitude' 和 'longitude' 列，跳过。")
            continue

        # 4) 按照经纬度进行内连接筛选
        merged_df = pd.merge(
            snow_df,
            grid_df,
            on=['latitude', 'longitude'],
            how='inner'
        )

        # 5) 生成输出文件名
        # 例如: snowAlbedo_1995.csv -> snowAlbedo_1995_grid.csv
        base_name = os.path.basename(snow_file)
        name, ext = os.path.splitext(base_name)
        output_file = os.path.join(snow_folder, f"{name}_grid{ext}")

        # 6) 写出匹配后的数据到新的 CSV 文件
        try:
            merged_df.to_csv(output_file, index=False, encoding='utf-8-sig')
            print(f"文件 {base_name} 处理完毕，匹配行数：{len(merged_df)}，输出文件：{output_file}")
        except Exception as e:
            print(f"写入文件 {output_file} 时出错：{e}")

if __name__ == "__main__":
    filter_all_snows_by_grid()


文件 snowAlbedo_1995.csv 处理完毕，匹配行数：2124，输出文件：F:\TUB\greenlandProject\ERA5LAND\snowalbedo\snowAlbedo_1995_grid.csv
文件 snowAlbedo_1996.csv 处理完毕，匹配行数：2124，输出文件：F:\TUB\greenlandProject\ERA5LAND\snowalbedo\snowAlbedo_1996_grid.csv
文件 snowAlbedo_1997.csv 处理完毕，匹配行数：2124，输出文件：F:\TUB\greenlandProject\ERA5LAND\snowalbedo\snowAlbedo_1997_grid.csv
文件 snowAlbedo_1998.csv 处理完毕，匹配行数：2124，输出文件：F:\TUB\greenlandProject\ERA5LAND\snowalbedo\snowAlbedo_1998_grid.csv
文件 snowAlbedo_1999.csv 处理完毕，匹配行数：2124，输出文件：F:\TUB\greenlandProject\ERA5LAND\snowalbedo\snowAlbedo_1999_grid.csv
文件 snowAlbedo_2000.csv 处理完毕，匹配行数：2124，输出文件：F:\TUB\greenlandProject\ERA5LAND\snowalbedo\snowAlbedo_2000_grid.csv
文件 snowAlbedo_2001.csv 处理完毕，匹配行数：2124，输出文件：F:\TUB\greenlandProject\ERA5LAND\snowalbedo\snowAlbedo_2001_grid.csv
文件 snowAlbedo_2002.csv 处理完毕，匹配行数：2124，输出文件：F:\TUB\greenlandProject\ERA5LAND\snowalbedo\snowAlbedo_2002_grid.csv
文件 snowAlbedo_2003.csv 处理完毕，匹配行数：2124，输出文件：F:\TUB\greenlandProject\ERA5LAND\snowalbedo\snowAlbedo_2003_g